In [1]:
import os
import pandas as pd
import tiktoken

In [2]:
blind_df = pd.read_csv("../data/processed/validation/blind_dataset_for_llm.csv")

enc = tiktoken.get_encoding("cl100k_base")

output_dir = "../data/processed/validation/batches"
os.makedirs(output_dir, exist_ok=True)

In [3]:
blind_df = blind_df.sort_values("anchor_id")

batch_num = 1
total_tokens_all = 0

for anchor_id, group in blind_df.groupby("anchor_id"):
    anchor_text = group.iloc[0]["anchor_text"]

    candidates_text = ""
    for _, row in group.iterrows():
        candidates_text += (
            f"ID: {row['candidate_id']}\n{row['candidate_text']}\n{'-' * 20}\n"
        )

    full_prompt = (
        f"=== ЦЕЛЕВАЯ ВАКАНСИЯ ===\n{anchor_text}\n\n"
        f"=== ПРЕДЛОЖЕННЫЕ ВАКАНСИИ ===\n{candidates_text}"
    )

    batch_tokens = len(enc.encode(full_prompt))
    total_tokens_all += batch_tokens

    file_path = os.path.join(output_dir, f"batch_{batch_num}.txt")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(full_prompt)

    batch_num += 1

print(f"Готово! Сформировано {batch_num - 1} файлов-батчей в папке {output_dir}/")
print(f"Общее количество токенов: ~{total_tokens_all}")

Готово! Сформировано 50 файлов-батчей в папке ../data/processed/validation/batches/
Общее количество токенов: ~2124881
